# DeepRacer-Genesis on Colab (T4)

AWS DeepRacer RL on [Genesis](https://genesis-world.readthedocs.io/) —
GPU-batched physics, TorchRL PPO, config-as-code experiments.

Everything runs directly in the cells: define an experiment class, train it
(~10 min on a T4, mostly one-time JIT), watch the trained cars, save the
checkpoint to your Drive.

Colab note: Colab VMs have no NVIDIA graphics userland, so the Madrona/Nyx
*camera-observation* renderers can't run here — feature-vector training runs
entirely on CUDA, and videos render through Mesa (software, slower).

In [ ]:
# @title GPU check
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
# @title Install (uv)
REPO = "https://github.com/Luna-v0/deepracer-genesis.git"
BRANCH = "main"

# Colab preinstalls TensorFlow, whose bundled LLVM clashes with Mesa's
# software renderer (segfault at first video frame). We don't use TF.
!pip uninstall -q -y tensorflow tensorflow-cpu tf-keras 2>/dev/null
!pip install -q uv
# --reinstall-package + --no-cache: after "Restart runtime" the VM disk
# survives — never trust a cached install of the package under development
!uv pip install --system -q --no-cache --reinstall-package deepracer-genesis \
    "deepracer-genesis @ git+{REPO}@{BRANCH}"

from deepracer_genesis.experiment import Experiment, run   # fail fast
from deepracer_genesis import ASSETS_DIR
import os
assert os.path.exists(os.path.join(ASSETS_DIR, "routes", "reinvent_base.npy")), "assets missing"
print("install OK")

## The experiment: one class, then `run()` it

Config as class attributes, the env / DR / policy pipeline as a `>>` chain.
Every run lands in `runs/{group}/{variant}-{seed}-{id}/` (content-hashed —
re-running an identical config is a cache hit) with TensorBoard logs,
checkpoints and an `eval_record.json`.

In [ ]:
from deepracer_genesis.experiment import (
    Experiment,
    FeatureEnvironment,
    VectorPolicy,
    run,
)


class ColabRacer(Experiment):
    # training configuration
    seed = 0
    total_env_steps = 5_000_000       # ~8 min at the T4's ~10k steps/s
    eval_every_steps = 1_000_000      # deterministic eval every 1M steps
    ablation_group = "colab"
    variant = "feature"

    # experiment hyperparameters (any name you like)
    num_envs = 512                    # T4-sized fleet
    lookahead_k = 10

    def pipeline(self):
        return (
            FeatureEnvironment(lookahead_k=self.lookahead_k,
                               num_envs=self.num_envs)
            >> VectorPolicy(keys=("state",))
        )

In [ ]:
# @title Train it (first run includes Genesis JIT compilation — a few minutes)
record = run(ColabRacer, root="/content/runs")
print("\nfinal eval:", {k: round(v, 3) for k, v in record.metrics.items()
                        if isinstance(v, (int, float))})
print("eval history:", [(h["frames"], round(h["completion_rate"], 2))
                        for h in record.eval_history])

In [ ]:
# @title Watch the trained cars (Mesa software rendering: ~35 s warmup, then ~0.4 s/frame)
from deepracer_genesis.experiment.visualize import rollout_video

mp4 = rollout_video("ColabRacer", root="/content/runs",
                    steps=300, num_envs=8,
                    spectator_res=(640, 480))   # software rendering: keep it modest

from IPython.display import Video
Video(mp4, embed=True, width=720)

In [ ]:
# @title Run report (aggregates every eval_record.json under runs/)
from deepracer_genesis.experiment.report import build_report
from IPython.display import Markdown
build_report("/content/runs", out_md="/content/runs/report.md")
Markdown(open("/content/runs/report.md").read())

## Save the trained model

The run directory is self-contained: `best.pt` (actor+critic weights +
the spec that trained them), `spec.json`, `eval_record.json`, TensorBoard
events, videos. Copy it to Google Drive (survives the session):

In [ ]:
# @title Save to Google Drive (set SAVE_TO_DRIVE = True; asks for authorization)
SAVE_TO_DRIVE = False

import glob, os, shutil
run_dir = sorted(glob.glob("/content/runs/colab/*"))[-1]
print("run dir:", run_dir)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = os.path.join("/content/drive/MyDrive/deepracer_runs",
                        os.path.basename(run_dir))
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print("saved:", dest)

In [ ]:
# @title ...or download the checkpoint directly (uncomment)
# from google.colab import files
# files.download(run_dir + "/best.pt")
print("checkpoint:", run_dir + "/best.pt")

Loading it later (any machine):

```python
from deepracer_genesis.experiment.visualize import rollout_video
rollout_video("ColabRacer", ckpt="path/to/best.pt", track="Monaco")
```

## Where to go from here

- **Variants**: `class Longer(ColabRacer): total_env_steps = 20_000_000` —
  each subclass gets its own content-addressed run directory.
- **More tracks**: `from deepracer_genesis.tools.track_builder import
  fetch_official_track; fetch_official_track("Vegas_track")` — then
  `tracks=("Vegas_track",)` in the env stage. Draw your own with
  `notebooks/track_designer.ipynb`.
- **Camera policies + DR** (`CameraEnvironment`,
  `DomainRandomizationTrackAppearance`, ...) need Madrona/Nyx — run those on
  a machine with NVIDIA Vulkan (not Colab).
- **Custom algorithms**: the `Algorithm` protocol in
  `deepracer_genesis/experiment/algorithms.py`.
- See `TUTORIAL.md` in the repo for the full walkthrough.